# Evaluación en BLD-AR

> Inferencia por tiles con SAHI sobre BLD-AR y métricas COCO en las clases de `docs/MAPEO_CLASES.md`.

In [ ]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

In [ ]:
%pip install ultralytics==8.4.173 sahi==0.12.8 pycocotools==2.0.11

## Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

## Rutas y parámetros

In [ ]:
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
BLDAR = REPO / "datasets/BLD-AR"
RUN_DIR = REPO / "runs/yolo26m-seg"
WEIGHTS = RUN_DIR / "weights/best.pt"

TILE = 1024
OVERLAP = 0.2
CONF = 0.05
DEVICE = "cuda:0"

assert WEIGHTS.exists(), f"No existen los pesos: {WEIGHTS}"

## Mapeo de clases

> Las demás clases de BLD-AR son zonas a ignorar: lo que se predice sobre ellas no cuenta como acierto ni como error.

In [ ]:
# Clase de FloorPlanCAD (predicción) -> clase de BLD-AR (etiqueta)
CLASS_MAP = {
    "wall": "wall", "single_door": "door", "window": "window", "sliding_door": "sliding_door",
    "stair": "stair", "elevator": "elevator", "railing": "railing",
}
CATEGORIES = {name: i + 1 for i, name in enumerate(sorted(set(CLASS_MAP.values())))}

## Etiquetas de BLD-AR en formato COCO

In [ ]:
def shape_points(shape):
    """Puntos de una forma LabelMe; los círculos se aproximan con 32 lados."""
    points = np.array(shape["points"], dtype=float)
    if shape["shape_type"] == "circle":
        center, edge = points
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        points = center + np.linalg.norm(edge - center) * np.stack([np.cos(angles), np.sin(angles)], axis=1)
    return points


def coco_annotation(points, image_id, category_id, iscrowd):
    x, y = points[:, 0], points[:, 1]
    return {
        "image_id": image_id, "category_id": category_id, "iscrowd": iscrowd,
        "segmentation": [points.ravel().tolist()],
        "bbox": [float(x.min()), float(y.min()), float(x.max() - x.min()), float(y.max() - y.min())],
        "area": float(0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))),
    }


images, annotations = [], []
for image_id, json_path in enumerate(sorted(BLDAR.glob("*.json")), start=1):
    data = json.loads(json_path.read_text())
    images.append({"id": image_id, "file_name": data["imagePath"],
                   "width": data["imageWidth"], "height": data["imageHeight"]})
    for shape in data["shapes"]:
        points = shape_points(shape)
        if shape["label"] in CATEGORIES:
            annotations.append(coco_annotation(points, image_id, CATEGORIES[shape["label"]], iscrowd=0))
        else:
            # Zona a ignorar (crowd de COCO), repetida para cada clase
            annotations += [coco_annotation(points, image_id, category_id, iscrowd=1)
                            for category_id in CATEGORIES.values()]
for annotation_id, annotation in enumerate(annotations, start=1):
    annotation["id"] = annotation_id

coco_gt = COCO()
coco_gt.dataset = {"images": images, "annotations": annotations,
                   "categories": [{"id": i, "name": name} for name, i in CATEGORIES.items()]}
coco_gt.createIndex()

## Modelo

In [ ]:
model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(WEIGHTS),
    confidence_threshold=CONF, device=DEVICE, image_size=TILE,
)

## Inferencia por tiles

In [ ]:
predictions = []
for image in images:
    result = get_sliced_prediction(
        str(BLDAR / image["file_name"]), model,
        slice_height=TILE, slice_width=TILE,
        overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
        perform_standard_pred=False, verbose=0,
        # Con confianza < 0.1 SAHI cambia a NMS y no une los muros cortados entre tiles
        force_postprocess_type=True,
    )
    for prediction in result.to_coco_predictions(image_id=image["id"]):
        name = CLASS_MAP.get(prediction["category_name"])
        if name is not None:
            prediction["category_id"] = CATEGORIES[name]
            predictions.append(prediction)
print(f"{len(predictions)} predicciones en {len(images)} planos")

## Métricas

In [ ]:
def evaluate(iou_type):
    """AP50 y AP50-95 por clase y en total, con COCOeval."""
    evaluator = COCOeval(coco_gt, coco_gt.loadRes(predictions), iou_type)
    evaluator.evaluate()
    evaluator.accumulate()
    precision = evaluator.eval["precision"][:, :, :, 0, -1]  # IoU x recall x clase (área "all", 100 detecciones)
    mean = lambda p: p[p > -1].mean() if (p > -1).any() else float("nan")
    per_class = {name: (mean(precision[0, :, k]), mean(precision[:, :, k])) for k, name in enumerate(CATEGORIES)}
    return mean(precision[0]), mean(precision), per_class


box, seg = evaluate("bbox"), evaluate("segm")
instances = {name: sum(a["category_id"] == i and not a["iscrowd"] for a in annotations) for name, i in CATEGORIES.items()}

display(Markdown(
    f"**BLD-AR · {RUN_DIR.name} · tile={TILE}**\n\n| | mAP50 | mAP50-95 |\n|---|---|---|\n"
    f"| Box | {box[0]:.3f} | {box[1]:.3f} |\n| Seg | {seg[0]:.3f} | {seg[1]:.3f} |"))
display(Markdown(
    "| Clase | Instancias | AP50 (B) | AP50-95 (B) | AP50 (S) | AP50-95 (S) |\n|---|---|---|---|---|---|\n"
    + "\n".join(f"| {name} | {instances[name]} | {box[2][name][0]:.3f} | {box[2][name][1]:.3f} "
                f"| {seg[2][name][0]:.3f} | {seg[2][name][1]:.3f} |" for name in CATEGORIES)))